In [3]:
import math     

import random   

from typing import Optional, Tuple  

 

import torch                    

import torch.nn as nn           

from torch.utils.data import DataLoader, TensorDataset  # For batching data

 

if torch.backends.mps.is_available():

    device = torch.device("mps")

elif torch.cuda.is_available():

    device = torch.device("cuda")

else:

    device = torch.device("cpu")

 

print(f"Using device: {device}")

 

print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu126


In [4]:
class SimpleRNN(nn.Module):

    

    def __init__(self, input_size: int, hidden_size: int, output_size: int):

        

        super().__init__()  

        

        self.hidden_size = hidden_size

        

       

        self.rnn = nn.RNN(

            input_size,      

            hidden_size,     

            batch_first=True 

        )

        self.fc = nn.Linear(hidden_size, output_size)

 

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        

        out, h_n = self.rnn(x)

        

        return self.fc(out)

 

def one_hot_encode(char: str, vocab: list) -> torch.Tensor:

    idx = vocab.index(char)

    t = torch.zeros(len(vocab))

    t[idx] = 1.0

    return t

 

VOCAB = list("abcdefghijklmnopqrstuvwxyz ")

VOCAB_SIZE = len(VOCAB)  # 27 characters

 

print(f"Vocabulary: {VOCAB}")

print(f"Vocabulary size: {VOCAB_SIZE}")

 

SEQ_LEN = 10   

HIDDEN = 32
print(f"\nOne-hot encoding of 'a': {one_hot_encode('a', VOCAB)}")

print(f"One-hot encoding of 'z': {one_hot_encode('z', VOCAB)}")

print(f"One-hot encoding of ' ': {one_hot_encode(' ', VOCAB)}")

Vocabulary: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ']
Vocabulary size: 27

One-hot encoding of 'a': tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.])
One-hot encoding of 'z': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 0.])
One-hot encoding of ' ': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 1.])


In [5]:
def make_toy_sequences(num_samples: int = 100) -> Tuple[torch.Tensor, torch.Tensor]:

   

    inputs, targets = [], []

    

    for _ in range(num_samples):

        start = random.randint(0, max(0, len(VOCAB) - SEQ_LEN - 1))

        

        seq_chars = [VOCAB[(start + i) % len(VOCAB)] for i in range(SEQ_LEN + 1)]

        inp = torch.stack([one_hot_encode(c, VOCAB) for c in seq_chars[:-1]])

        tgt_idx = VOCAB.index(seq_chars[-1])

        inputs.append(inp)

        targets.append(tgt_idx)

    

    return torch.stack(inputs), torch.tensor(targets, dtype=torch.long)

 

model_rnn = SimpleRNN(VOCAB_SIZE, HIDDEN, VOCAB_SIZE).to(device)

In [6]:
print("Model architecture:")

print(model_rnn)

 

# Count parameters

total_params = sum(p.numel() for p in model_rnn.parameters())

print(f"\nTotal parameters: {total_params:,}")

 

# Create training data

X_toy, y_toy = make_toy_sequences(200)

print(f"\nTraining data shapes:")

print(f"  Inputs: {X_toy.shape}  (samples, sequence_length, vocab_size)")

print(f"  Targets: {y_toy.shape} (samples,)")

 

dataset = TensorDataset(X_toy, y_toy)

loader = DataLoader(dataset, batch_size=16, shuffle=True)

 

optimizer = torch.optim.Adam(model_rnn.parameters(), lr=1e-3)

 

criterion = nn.CrossEntropyLoss()

 

# Show one example

print("\nExample training sequence:")

example_chars = [VOCAB[X_toy[0, i].argmax().item()] for i in range(SEQ_LEN)]

target_char = VOCAB[y_toy[0].item()]

print(f"  Input: {''.join(example_chars)}")

print(f"  Target (next char): {target_char}")

Model architecture:
SimpleRNN(
  (rnn): RNN(27, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=27, bias=True)
)

Total parameters: 2,843

Training data shapes:
  Inputs: torch.Size([200, 10, 27])  (samples, sequence_length, vocab_size)
  Targets: torch.Size([200]) (samples,)

Example training sequence:
  Input: ghijklmnop
  Target (next char): q


In [7]:
print("Training the RNN...\n")

for epoch in range(3):

    model_rnn.train()

    total_loss = 0.0  # Track loss for this epoch

    for batch_x, batch_y in loader:

        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        

        optimizer.zero_grad()

        logits = model_rnn(batch_x)

        last_logits = logits[:, -1, :]

        loss = criterion(last_logits, batch_y)

        loss.backward()

        # STEP 6: Update weights using gradients

        optimizer.step()

        total_loss += loss.item()

    

    # Print epoch summary

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

print("\n Training complete!")

print("\nTesting on a few examples:")

model_rnn.eval()  # Set to evaluation mode

with torch.no_grad():  # Don't compute gradients for testing

    for i in range(5):

        # Get input sequence

        input_seq = X_toy[i:i+1].to(device)  # Keep batch dimension

        # Get prediction

        logits = model_rnn(input_seq)[:, -1, :]  # Last time step

        pred_idx = logits.argmax(dim=-1).item()

        input_chars = ''.join([VOCAB[X_toy[i, j].argmax().item()] for j in range(SEQ_LEN)])

        pred_char = VOCAB[pred_idx]

        true_char = VOCAB[y_toy[i].item()]

        

        status = "✓" if pred_char == true_char else "✗"

        print(f"  '{input_chars}' → Predicted: '{pred_char}', Actual: '{true_char}' {status}")

Training the RNN...

Epoch 1, Loss: 3.2537
Epoch 2, Loss: 3.1350
Epoch 3, Loss: 2.9999

 Training complete!

Testing on a few examples:
  'ghijklmnop' → Predicted: 'y', Actual: 'q' ✗
  'abcdefghij' → Predicted: 'k', Actual: 'k' ✓
  'fghijklmno' → Predicted: 'p', Actual: 'p' ✓
  'pqrstuvwxy' → Predicted: 'z', Actual: 'z' ✓
  'qrstuvwxyz' → Predicted: ' ', Actual: ' ' ✓
